### Implementation with 300 seconds timer

In [ ]:
import json
import sqlite3
import time
from datetime import datetime
from pathlib import Path
import requests

DB_PATH = Path.cwd() / "polymarket_gamma_dynamic.sqlite"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    return conn

def gamma_scraper(endpoint="markets", fetch_all=True, **kwargs):
    # ... [Keep your existing dynamic gamma_scraper function here] ...
    base_url = f"https://gamma-api.polymarket.com/{endpoint}"
    params = {
        "limit": kwargs.get("limit", 100),
        "offset": kwargs.get("offset", 0),
        "active": str(kwargs.get("active", "true")).lower(),
        "closed": str(kwargs.get("closed", "false")).lower(),
    }
    params.update(kwargs)

    try:
        if not fetch_all:
            response = requests.get(base_url, params=params)
            response.raise_for_status()
            return response.json()

        all_items = []
        limit = int(params.get("limit", 100))
        offset = int(params.get("offset", 0))

        while True:
            current_params = dict(params)
            current_params["limit"] = limit
            current_params["offset"] = offset

            response = requests.get(base_url, params=current_params)
            response.raise_for_status()
            page_data = response.json()

            if isinstance(page_data, list):
                all_items.extend(page_data)
                if len(page_data) < limit:
                    break
                offset += limit
            else:
                return page_data

        return all_items
    except Exception as e:
        print(f"Error fetching {endpoint}: {e}")
        return []

def save_pipeline_dynamic(conn, data, table_name="markets"):
    """
    Dynamically creates/updates a SQLite table. 
    Now stores data sequentially by appending a timestamp.
    """
    if not data:
        return
        
    # 1. INJECT TIMESTAMP: Record the exact time of the scrape
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    for item in data:
        item["scraped_at"] = current_time

    cursor = conn.cursor()
    all_keys = set()
    for item in data:
        all_keys.update(item.keys())
    
    columns = sorted(list(all_keys))
    
    # Force "id" and "scraped_at" to be the first two columns for clean formatting
    if "scraped_at" in columns:
        columns.remove("scraped_at")
    if "id" in columns:
        columns.remove("id")
    columns.insert(0, "id")
    columns.insert(1, "scraped_at")

    # 2. CREATE TABLE WITH COMPOSITE PRIMARY KEY
    col_defs = []
    for col in columns:
        col_defs.append(f'"{col}" TEXT')
        
    # We define the Primary Key at the end, combining 'id' and 'scraped_at'
    create_table_sql = f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            {', '.join(col_defs)},
            PRIMARY KEY ("id", "scraped_at")
        )
    """
    cursor.execute(create_table_sql)

    # 3. Handle Schema Evolution
    cursor.execute(f"PRAGMA table_info({table_name})")
    existing_columns = {row[1] for row in cursor.fetchall()}
    
    for col in columns:
        if col not in existing_columns:
            cursor.execute(f'ALTER TABLE {table_name} ADD COLUMN "{col}" TEXT')

    # 4. Insert the sequential data
    placeholders = ", ".join(["?"] * len(columns))
    col_names = ", ".join([f'"{col}"' for col in columns])
    # INSERT OR IGNORE prevents errors if the script accidentally triggers twice in the same exact second
    insert_sql = f"INSERT OR IGNORE INTO {table_name} ({col_names}) VALUES ({placeholders})"

    rows_to_insert = []
    for item in data:
        row = []
        for col in columns:
            val = item.get(col)
            # Convert dicts, lists, and booleans into JSON strings
            if isinstance(val, (dict, list, bool)):
                row.append(json.dumps(val))
            else:
                row.append(val)
        rows_to_insert.append(tuple(row))

    cursor.executemany(insert_sql, rows_to_insert)
    conn.commit()


def run_job(keyword="gold", limit_results=None):
    """
    Executes a single scraping run with keyword and tag filters.
    """
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    limit_text = "All" if limit_results is None else limit_results
    print(f"\n[{current_time}] Starting scrape (Keyword: '{keyword}', Limit: {limit_text})...")
    
    try:
        conn = init_db()
        
        # 1. Fetch tags to dynamically map categories
        print("Fetching tags to map categories...")
        # FIX: Back to limit=100. Polymarket caps at 100; higher numbers break pagination.
        all_tags = gamma_scraper(endpoint="tags", limit=100)
        target_keywords = ["finance", "crypto", "geopolitics", "politics"]
        
        target_tag_ids = []
        for tag in all_tags:
            label = str(tag.get("label", "")).lower()
            slug = str(tag.get("slug", "")).lower()
            if any(tk in label or tk in slug for tk in target_keywords):
                target_tag_ids.append(tag.get("id"))
                
        # Deduplicate tag IDs
        target_tag_ids = list(set(target_tag_ids))
        save_pipeline_dynamic(conn, all_tags, "tags")
        print(f"Matched {len(target_tag_ids)} relevant tag IDs. Fetching markets...")
        
        # 2. Fetch all markets for these specific tags
        raw_markets = {} 
        
        for t_id in target_tag_ids:
            # FIX: Back to limit=100 here as well for safety.
            tag_markets = gamma_scraper(endpoint="markets", tag_id=t_id, limit=100)
            for m in tag_markets:
                raw_markets[m["id"]] = m
                
        market_list = list(raw_markets.values())
        print(f"Found {len(market_list)} unique active markets across targeted tags.")
        
        # 3. Apply Keyword Filter
        filtered_markets = []
        if keyword:
            keyword_lower = keyword.lower()
            for m in market_list:
                question = str(m.get("question", "")).lower()
                description = str(m.get("description", "")).lower()
                
                if keyword_lower in question or keyword_lower in description:
                    filtered_markets.append(m)
        else:
            filtered_markets = market_list
            
        # 4. Sort by Volume & Apply Optional Limit
        filtered_markets.sort(key=lambda x: float(x.get("volume", 0) or 0), reverse=True)
        
        if limit_results is not None:
            filtered_markets = filtered_markets[:limit_results]
            
        # 5. Save to Database
        save_pipeline_dynamic(conn, filtered_markets, "markets")
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Success: Saved {len(filtered_markets)} matching markets.")
        
    except Exception as e:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] CRITICAL ERROR during scrape: {e}")
        
    finally:
        if 'conn' in locals():
            conn.close()




In [ ]:
'''infinite loop to run the scraper indefinitely'''
if __name__ == "__main__":
    INTERVAL_SECONDS = 300 # 5 minutes
    
    print(f"Initializing Wall-Clock Scraper (Interval: {INTERVAL_SECONDS}s).")
    print("Press Ctrl+C to stop.")

    while True:
        # 1. Calculate how long to sleep to hit the next 5-minute mark
        now = time.time()
        # % operator finds the seconds past the last 5-minute mark
        seconds_past_interval = now % INTERVAL_SECONDS
        time_to_wait = INTERVAL_SECONDS - seconds_past_interval
        
        # 2. Print the countdown for clarity
        next_run = datetime.fromtimestamp(now + time_to_wait).strftime('%H:%M:%S')
        print(f"Waiting {int(time_to_wait)}s until next scheduled run at {next_run}...")
        
        time.sleep(time_to_wait)
        
        # 3. Execute the job
        run_job()

In [ ]:
'''test single scraper run'''
run_job(keyword="gold", limit_results=10)


[2026-03-29 15:02:52] Starting scrape (Keyword: 'gold', Limit: 10)...
Fetching tags to map categories...
Matched 34 relevant tag IDs. Fetching markets...
Found 13271 unique active markets across targeted tags.
[15:03:38] Success: Saved 10 matching markets.


In [ ]:
'''test print of market title, volume and scraping time'''


import sqlite3
import json
from pathlib import Path

DB_PATH = Path.cwd() / "polymarket_gamma_dynamic.sqlite"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Removed 'tags' from the SELECT statement
cursor.execute("SELECT question, volume, scraped_at FROM markets LIMIT 10;")
results = cursor.fetchall()

print(f"Found {len(results)} markets. Here is the breakdown:\n")
print("-" * 50)

for row in results:
    question = row[0]
    # Format volume to look like a readable dollar amount
    volume = f"${float(row[1]):,.2f}" if row[1] else "$0.00"
    scraped_at = row[2]

    print(f"Market: {question}")
    print(f"Volume: {volume}")
    print(f"Scraped At: {scraped_at}")
    print("-" * 50)

conn.close()

Found 10 markets. Here is the breakdown:

--------------------------------------------------
Market: Will Gold (GC) hit (HIGH) $5,500 by end of June?
Volume: $787,189.15
Scraped At: 2026-03-29 14:59:30
--------------------------------------------------
Market: Will Gold (GC) hit (LOW) $3,000 by end of March?
Volume: $489,368.97
Scraped At: 2026-03-29 14:59:30
--------------------------------------------------
Market: Will Bitcoin outperform Gold in 2026?
Volume: $376,976.33
Scraped At: 2026-03-29 14:59:30
--------------------------------------------------
Market: Will Bitcoin have the best performance in 2026?
Volume: $370,990.59
Scraped At: 2026-03-29 14:59:30
--------------------------------------------------
Market: Will Gold (GC) hit (HIGH) $7,000 by end of March?
Volume: $365,473.67
Scraped At: 2026-03-29 14:59:30
--------------------------------------------------
Market: Will Gold (GC) hit (HIGH) $10,000 by end of March?
Volume: $324,555.21
Scraped At: 2026-03-29 14:59:30
-------